# PhyAI Pokedex - train a 4-Pokemon classifier and export it for the UNO Q

Bulbasaur / Charizard / Pikachu / Squirtle - trained here on a free Colab GPU,
then exported to a small `.tflite` file you copy onto shimi and run offline.
No internet needed at inference time; all the heavy lifting happens once, here.

**How to run this notebook:**
1. Runtime -> Change runtime type -> **T4 GPU** (free tier is fine)
2. Run every cell top to bottom
3. When asked, upload a **zip** of your 4 folders (see Step 2 below)
4. At the end, download `pokedex_phyai.tflite` and `labels.txt`

**Why this approach:** with only ~40 photos per Pokemon, training a model from
scratch would badly overfit. Instead we start from **MobileNetV2**, a small
model already trained on a million general photos, and only retrain its last
few layers to tell your 4 Pokemon apart. This is called *transfer learning* -
it needs far less data and trains in minutes, not hours.

## Step 1 - install & imports

In [ ]:
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

import os, shutil, zipfile, glob, random
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

## Step 2 - upload your dataset

Zip your 4 folders into ONE file first, structured like this:

```
pokemon_data.zip
  bulbasaur/   (your ~40 images)
  charizard/   (your ~40 images)
  pikachu/     (your ~40 images)
  squirtle/    (your ~40 images)
```

Folder names become the class labels, so keep them lowercase and simple -
exactly as above. Run the cell below, then click "Choose Files" and pick
your zip.

In [ ]:
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print('Uploaded:', zip_name)

DATA_DIR = '/content/pokemon_data'
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(DATA_DIR)

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(DATA_DIR)

# Sometimes a zip extracts into a nested subfolder (e.g. if you zipped a
# folder rather than its contents) - flatten that automatically.
entries = [e for e in os.listdir(DATA_DIR) if not e.startswith('.')]
if len(entries) == 1 and os.path.isdir(os.path.join(DATA_DIR, entries[0])):
    nested = os.path.join(DATA_DIR, entries[0])
    for item in os.listdir(nested):
        shutil.move(os.path.join(nested, item), DATA_DIR)
    shutil.rmtree(nested)

class_names = sorted([d for d in os.listdir(DATA_DIR)
                       if os.path.isdir(os.path.join(DATA_DIR, d))])
print('\nFound classes:', class_names)
for c in class_names:
    n = len(glob.glob(os.path.join(DATA_DIR, c, '*')))
    print(f'  {c}: {n} images')

assert len(class_names) >= 2, 'Need at least 2 class folders - check your zip structure.'

## Step 3 - look at a few images

Quick sanity check before spending GPU time - make sure the right photos
landed in the right folders.

In [ ]:
fig, axes = plt.subplots(1, len(class_names), figsize=(4*len(class_names), 4))
if len(class_names) == 1:
    axes = [axes]
for ax, c in zip(axes, class_names):
    imgs = glob.glob(os.path.join(DATA_DIR, c, '*'))
    img = tf.keras.utils.load_img(random.choice(imgs))
    ax.imshow(img)
    ax.set_title(c)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Step 4 - build the datasets

`IMG_SIZE = 160` keeps the model light enough for the UNO Q. We hold back
20% of images for validation, so we can tell if the model is actually
learning or just memorizing.

In [ ]:
IMG_SIZE = 160
BATCH_SIZE = 8   # small dataset -> small batches

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_ds.class_names   # keep this order - it defines label indices
print('Class order (this is the order the model will output):', class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

## Step 5 - data augmentation

With only ~30 training images per class after the validation split, the
model can easily just memorize your exact photos instead of learning what
each Pokemon actually looks like. Augmentation manufactures variety - random
flips, rotations, zooms, brightness shifts - so every epoch the model sees
slightly different versions of the same images. This matters more here than
in a normal-sized dataset.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomContrast(0.2),
], name='augmentation')

# preview what augmentation does to one image
for images, _ in train_ds.take(1):
    plt.figure(figsize=(10, 10))
    first = images[0]
    for i in range(9):
        ax = plt.subplot(3, 3, i+1)
        augmented = data_augmentation(tf.expand_dims(first, 0), training=True)
        plt.imshow(augmented[0].numpy().astype('uint8'))
        plt.axis('off')
    plt.suptitle('9 augmented versions of the same photo')
    plt.show()

## Step 6 - build the model (MobileNetV2 transfer learning)

MobileNetV2 already knows how to see edges, textures, shapes and colours
from training on a million photos. We freeze that knowledge and only train a
small classifier head on top that maps its understanding onto YOUR 4
Pokemon. This is why it works with so little data.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False   # freeze - keep its learned features intact

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)   # helps prevent overfitting on a small dataset
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Step 7 - train (phase 1: frozen base)

Only the small classifier head trains here - fast, and safe from overfitting
since most of the model is frozen.

In [ ]:
EPOCHS_PHASE1 = 15

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=5, restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    callbacks=[early_stop]
)

## Step 8 - fine-tune (phase 2: unfreeze the top layers)

Now that the classifier head is trained, we carefully unfreeze the LAST few
layers of MobileNetV2 and train with a much smaller learning rate. This lets
the model adjust its higher-level features specifically for Pokemon shapes
and colours, without wrecking everything it already knows.

In [ ]:
base_model.trainable = True

# freeze everything except the last ~20 layers
FINE_TUNE_AT = len(base_model.layers) - 20
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),  # much smaller now
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

EPOCHS_PHASE2 = 10
total_epochs = EPOCHS_PHASE1 + EPOCHS_PHASE2

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=total_epochs,
    initial_epoch=history.epoch[-1] + 1,
    callbacks=[early_stop]
)

## Step 9 - check the results

With this little data, don't expect 99% - if validation accuracy lands
somewhere around 70-90% that's a realistic, usable result for 4 visually
distinct Pokemon. If it's much lower, the fixes are: more photos per class,
or photos with more varied backgrounds/lighting/angles.

In [ ]:
acc = history.history['accuracy'] + history_fine.history['accuracy']
val_acc = history.history['val_accuracy'] + history_fine.history['val_accuracy']
loss = history.history['loss'] + history_fine.history['loss']
val_loss = history.history['val_loss'] + history_fine.history['val_loss']

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='train accuracy')
plt.plot(val_acc, label='validation accuracy')
plt.axvline(EPOCHS_PHASE1-1, color='gray', linestyle='--', label='fine-tuning starts')
plt.legend(); plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(loss, label='train loss')
plt.plot(val_loss, label='validation loss')
plt.axvline(EPOCHS_PHASE1-1, color='gray', linestyle='--', label='fine-tuning starts')
plt.legend(); plt.title('Loss')
plt.show()

final_val_acc = val_acc[-1]
print(f'\nFinal validation accuracy: {final_val_acc*100:.1f}%')
if final_val_acc < 0.6:
    print('This is low - consider adding more photos per Pokemon, ideally with')
    print('varied backgrounds, lighting, and angles, then re-run from Step 2.')

## Step 10 - confusion matrix

Shows exactly which Pokemon the model mixes up, if any - more useful than a
single accuracy number.

In [ ]:
y_true = []
y_pred = []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

cm = tf.math.confusion_matrix(y_true, y_pred, num_classes=len(class_names)).numpy()

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xticks(range(len(class_names)), class_names, rotation=45)
plt.yticks(range(len(class_names)), class_names)
plt.xlabel('Predicted'); plt.ylabel('Actual')
for i in range(len(class_names)):
    for j in range(len(class_names)):
        plt.text(j, i, cm[i, j], ha='center', va='center',
                 color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.title('Confusion Matrix (validation set)')
plt.tight_layout()
plt.show()

## Step 11 - export to TFLite

Converts and quantizes the model to int8 - this is what makes it small and
fast enough to run on the UNO Q's CPU. Quantization can cost a little
accuracy, which is why we checked the float accuracy first, above.

In [ ]:
def representative_dataset():
    """A handful of real images the converter uses to calibrate the int8
    quantization ranges. Needs real data, not random noise, or the
    quantized model performs badly."""
    for images, _ in train_ds.take(5):
        for i in range(images.shape[0]):
            img = tf.expand_dims(images[i], 0)
            yield [tf.cast(img, tf.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_model = converter.convert()

TFLITE_PATH = '/content/pokedex_phyai.tflite'
with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

size_kb = os.path.getsize(TFLITE_PATH) / 1024
print(f'Saved: {TFLITE_PATH}  ({size_kb:.0f} KB)')

# labels.txt - one class name per line, IN ORDER. The UNO Q code reads this
# to turn the model's output index back into a Pokemon name.
LABELS_PATH = '/content/labels.txt'
with open(LABELS_PATH, 'w') as f:
    f.write('\n'.join(class_names))
print(f'Saved: {LABELS_PATH}')
print('\nClass order:', class_names)

## Step 12 - sanity check the exported TFLite model

Runs a few validation images through the *exported* (quantized) model, not
the original, to confirm the conversion didn't break anything before you
download it.

In [ ]:
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print('Input :', input_details[0]['shape'], input_details[0]['dtype'])
print('Output:', output_details[0]['shape'], output_details[0]['dtype'])

correct = 0
total = 0
for images, labels in val_ds.take(3):
    for i in range(images.shape[0]):
        img = tf.cast(images[i], tf.uint8).numpy()
        img = np.expand_dims(img, 0)
        interpreter.set_tensor(input_details[0]['index'], img)
        interpreter.invoke()
        out = interpreter.get_tensor(output_details[0]['index'])[0]
        pred_idx = np.argmax(out)
        true_idx = np.argmax(labels[i].numpy())
        correct += int(pred_idx == true_idx)
        total += 1

print(f'\nQuantized TFLite model: {correct}/{total} correct on a sample of validation images')

## Step 13 - download

Downloads both files. You need both - the model gives you a class number,
`labels.txt` turns that back into a Pokemon name.

In [ ]:
files.download(TFLITE_PATH)
files.download(LABELS_PATH)

## Next: deploying on shimi

1. Copy `pokedex_phyai.tflite` and `labels.txt` to `~/pokedexv5/` on shimi
2. `pip3 install tflite-runtime --break-system-packages` (a small package -
   this does NOT pull in full TensorFlow, which would fill the disk)
3. Ask Claude for the inference code that wires this into the PhyAI
   Challenge menu item - it needs the exact input size (160x160) and
   preprocessing this notebook used, which is why the two files travel
   together.

Come back to this notebook any time you add more photos (like the 5th
"unknown" class) - just re-run from Step 2 with a new zip.